# Chapter 1 companion: evolving a neural controller

This original **nevo** tutorial makes the idea of evolving neural-network behavior concrete. It is inspired by Chapter 1, *Introduction*, of [Neuroevolution: Harnessing Creativity in AI Agent Design](https://neuroevolutionbook.com/), by Sebastian Risi, Yujin Tang, David Ha, and Risto Miikkulainen.

The supplied Chapter 1 PDF was checked on 2026-11-24. Section 1.1 (pp. 3–5) describes keeping the best 20 of 100 neural weight vectors and producing 80 offspring through crossover and mutation. Our original first example simplifies this to one retained controller and mutation only. Section 6 adds the 100/20/80 population structure using the public GA. The listed coding assignments begin in Chapter 2; this notebook remains independent companion material.

We will evolve three numbers that control a moving point. There are no correct-action labels and no backpropagation. An entire trajectory determines how good a controller is.

**Run from top to bottom with the Rust/Evcxr kernel.** Start Jupyter in the repository's notebooks directory; see [notebook setup](README.md).

In [1]:
:dep nevo = { path = "../crates/nevo" }
:dep plotters = { version = "=0.3.7", default-features = false, features = ["evcxr", "line_series"] }

In [2]:
use nevo::algorithms::{ElitistConfig, ElitistSearch, Objective};
use nevo::tasks::TargetReach;
use nevo::step;
use plotters::prelude::*;

## 1. Genotype, controller, and environment

The genotype is the vector $[w_e,w_v,b]$. Its phenotype is a single tanh neuron, used repeatedly as a controller:

$$a_t=\tanh\!\left(w_e(-x_t)+w_vv_t+b\right).$$

Here, $x_t$ is position relative to the target at zero, $v_t$ is velocity, and $a_t$ is an acceleration command. The environment advances using

$$v_{t+1}=0.9v_t+0.2a_t,\qquad x_{t+1}=x_t+0.1v_{t+1}.$$

Each episode starts at zero velocity and lasts $T=80$ steps. The training starts are $S=\{-1,-0.5,0.5,1\}$. We **minimize**

$$L=\frac{1}{|S|T}\sum_{s\in S}\sum_{t=0}^{T-1}
\left(x_{t+1}^2+0.01a_t^2\right).$$

The position term rewards reaching the target, while the action term discourages unnecessary effort. Each evaluation resets the environment. These equations and numerical settings belong to this original teaching task.

In [3]:
let mut task = TargetReach::default();
let mut config = ElitistConfig::new(TargetReach::PARAMETERS, 42);
config.objective = Objective::Minimize;
let mut search = ElitistSearch::new(config)?;
let baseline = task.value(&[0.0; 3])?;
let mut history: Vec<(usize, f64)> = Vec::new();

println!("Training starts: {:?}", task.starts());
println!("Steps per episode: {}", task.steps());
println!("Search seed: {}", search.config().seed);
println!("Candidates per batch: {}", search.config().offspring);
println!("No-action baseline loss: {baseline:.9}");

Training starts: [-1.0, -0.5, 0.5, 1.0]


Steps per episode: 80


Search seed: 42


Candidates per batch: 32


No-action baseline loss: 0.625000000


## 2. Generate → Evaluate → Select

Each call to the public **step** helper executes one complete batch:

1. **Generate:** the initial batch samples 32 controllers. Later batches perturb the retained controller's three genes.
2. **Evaluate:** each candidate runs from all four training starts.
3. **Select:** retain the candidate with the lowest loss, including the previous incumbent, and advance the search state.

In nevo, *Select* includes the state changes needed for the next cycle. Other algorithms can use this phase for archive insertion or distribution adaptation.

ElitistSearch is a small mutation-only teaching search with uniform perturbations. It is not a canonical GA or a Gaussian evolution strategy. It caches the incumbent's score, so this configuration is intended for the deterministic task used here.

The next cell runs **100 batches total, including initialization**. It calls the library implementation; no optimizer is reimplemented in this notebook.

In [4]:
for _ in 0..100 {
    // Generate -> Evaluate -> Select, through the public serial runner.
    step(&mut search, &mut task)?;
    let selected = search.best().ok_or("selection returned no controller")?;
    history.push((search.evaluations(), selected.output));
}

let best = search.best().cloned().ok_or("search has no selected controller")?;
let episodes = search.evaluations() * task.starts().len();
let simulation_steps = episodes * task.steps();

println!("Completed batches: {}", search.batches());
println!("Search evaluations: {}", search.evaluations());
println!("Training episodes: {episodes}");
println!("Training simulation steps: {simulation_steps}");
println!("Best genes [error, velocity, bias]: {:?}", best.job.input);
println!("Best training loss: {:.9}", best.output);

Completed batches: 100


Search evaluations: 3200


Training episodes: 12800


Training simulation steps: 1024000


Best genes [error, velocity, bias]: [8.277480418360419, -2.140629632961784, 0.0017547022593667558]


Best training loss: 0.036473747


## 3. Inspect the learning curve

The horizontal axis counts **objective evaluations**, so changing the population size does not hide its computational cost. The curve records the best retained training score after each batch.

An elitist best-so-far curve cannot increase on this deterministic task. That property does not imply a global optimum or a particular convergence rate.

The no-action baseline is printed above. It is omitted from the plot so the improvement from the initial random population remains visible.

In [5]:
let fig_learning = plotters::prelude::evcxr_figure((820, 420), |root| {
    root.fill(&WHITE)?;
    let y_max = history.iter().map(|(_, loss)| *loss)
        .fold(0.0, f64::max).max(0.001) * 1.08;
    let mut chart = ChartBuilder::on(&root)
        .caption("Neural-controller evolution", ("sans-serif", 22))
        .margin(16)
        .x_label_area_size(42)
        .y_label_area_size(55)
        .build_cartesian_2d(0_u32..search.evaluations() as u32, 0.0_f64..y_max)?;
    chart.configure_mesh()
        .x_desc("Search evaluations")
        .y_desc("Training loss (lower is better)")
        .draw()?;
    chart.draw_series(LineSeries::new(
        history.iter().map(|(evaluations, loss)| (*evaluations as u32, *loss)),
        BLUE.stroke_width(3),
    ))?.label("Best retained controller")
        .legend(|(x, y)| PathElement::new(vec![(x, y), (x + 22, y)], BLUE.stroke_width(3)));
    chart.configure_series_labels()
        .background_style(WHITE.mix(0.9))
        .border_style(BLACK)
        .draw()?;
    root.present()?;
    Ok(())
});
fig_learning

Neural-controller evolution
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Training loss (lower is better)
 
 
Search evaluations
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
0.0
 
 
 
0.01
 
 
 
0.02
 
 
 
0.03
 
 
 
0.04
 
 
 
0.05
 
 
 
0.06
 
 
 
 
0
 
 
 
500
 
 
 
1000
 
 
 
1500
 
 
 
2000
 
 
 
2500
 
 
 
3000
 
 
<polyline fill="none" opacity="1" stroke="#0000FF" stroke-width="3" points="78,67 85,118 92,127 100,143 107,150 114,153 122,162 129,163 136,166 144,168 151,174 158,175 166,178 173,179 180,180 188,180 195,181 202,182 210,182 217,182 224,184 232,185 239,185 246,186 254,187 261,187 268,187 275,188 283,188 290,188 297,188 305,188 312,188 319,188 327,188 334,188 341,189 349,189 356,189 363,189 371,189 378,189 385,189 393,190 400,190 407,190 415,190 422,190 429,190 437,190 444,190 451,190 458,190 466,190 473,190 480,190 488,190 495,190 502,190 510,190 517,190 524,190 532,190 539,190 546,190 554,190 561,190 568,190 576,190 583,191 590,191 598,191 605,191 612,191 620,191 627,191 634,191 641,191 649,191 656,191 663,191 671,191 678,191 685,191 693,191 700,191 707,191 715,191 722,191 729,191 737,191 744,191 751,191 759,191 766,191 773,191 781,191 788,191 795,191 803,191 "/>
 
 
 
Best retained controller

## 4. Freeze the controller, then inspect new starts

Only now do we evaluate starting positions $\{-1.25,-0.75,0.75,1.25\}$. These positions were not consulted during selection. The controller stays fixed.

This is a small held-out diagnostic, not evidence of broad generalization. Its raw loss is not directly comparable with the training loss because the starting positions differ.

The plots below show the two outer starts, comparing the evolved controller with an all-zero, no-action controller.

In [6]:
let held_out = TargetReach::new(vec![-1.25, -0.75, 0.75, 1.25], task.steps())?;
let held_out_loss = held_out.value(&best.job.input)?;
println!("Held-out starts: {:?}", held_out.starts());
println!("Held-out loss: {held_out_loss:.9}");

let starts_to_plot = [-1.25_f64, 1.25_f64];
let baseline_traces = starts_to_plot.iter()
    .map(|&start| task.rollout(&[0.0; 3], start))
    .collect::<Result<Vec<_>, _>>()?;
let evolved_traces = starts_to_plot.iter()
    .map(|&start| task.rollout(&best.job.input, start))
    .collect::<Result<Vec<_>, _>>()?;

Held-out starts: [-1.25, -0.75, 0.75, 1.25]


Held-out loss: 0.070859762


In [7]:
let fig_trajectories = plotters::prelude::evcxr_figure((820, 460), |root| {
    root.fill(&WHITE)?;
    let (minimum, maximum) = baseline_traces.iter().chain(&evolved_traces)
        .flat_map(|rollout| &rollout.states)
        .fold((0.0_f64, 0.0_f64), |(low, high), state| {
            (low.min(state.position), high.max(state.position))
        });
    let padding = (maximum - minimum).max(0.1) * 0.1;
    let mut chart = ChartBuilder::on(&root)
        .caption("Held-out trajectories with a fixed controller", ("sans-serif", 22))
        .margin(16)
        .x_label_area_size(42)
        .y_label_area_size(55)
        .build_cartesian_2d(0_u32..task.steps() as u32, (minimum - padding)..(maximum + padding))?;
    chart.configure_mesh()
        .x_desc("Simulation step")
        .y_desc("Position (target = 0)")
        .draw()?;
    chart.draw_series(LineSeries::new(
        [(0_u32, 0.0), (task.steps() as u32, 0.0)],
        BLACK.mix(0.3),
    ))?;
    for index in 0..starts_to_plot.len() {
        let color = [BLUE, RED][index];
        let start = starts_to_plot[index];
        chart.draw_series(LineSeries::new(
            baseline_traces[index].states.iter().map(|state| (state.step as u32, state.position)),
            color.mix(0.3).stroke_width(2),
        ))?.label(format!("No action, start {start:+.2}"))
            .legend(move |(x, y)| PathElement::new(vec![(x, y), (x + 22, y)], color.mix(0.3).stroke_width(2)));
        chart.draw_series(LineSeries::new(
            evolved_traces[index].states.iter().map(|state| (state.step as u32, state.position)),
            color.stroke_width(3),
        ))?.label(format!("Evolved, start {start:+.2}"))
            .legend(move |(x, y)| PathElement::new(vec![(x, y), (x + 22, y)], color.stroke_width(3)));
    }
    chart.configure_series_labels()
        .background_style(WHITE.mix(0.9))
        .border_style(BLACK)
        .draw()?;
    root.present()?;
    Ok(())
});
fig_trajectories

Held-out trajectories with a fixed controller
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Position (target = 0)
 
 
Simulation step
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-1.5
 
 
 
-1.0
 
 
 
-0.5
 
 
 
0.0
 
 
 
0.5
 
 
 
1.0
 
 
 
1.5
 
 
 
 
0
 
 
 
10
 
 
 
20
 
 
 
30
 
 
 
40
 
 
 
50
 
 
 
60
 
 
 
70
 
 
 
80
 
 
 
<polyline fill="none" opacity="0.3" stroke="#0000FF" stroke-width="2" points="71,372 80,372 89,372 98,372 107,372 116,372 125,372 135,372 144,372 153,372 162,372 171,372 180,372 189,372 199,372 208,372 217,372 226,372 235,372 244,372 254,372 263,372 272,372 281,372 290,372 299,372 308,372 318,372 327,372 336,372 345,372 354,372 363,372 372,372 382,372 391,372 400,372 409,372 418,372 427,372 437,372 446,372 455,372 464,372 473,372 482,372 491,372 501,372 510,372 519,372 528,372 537,372 546,372 555,372 565,372 574,372 583,372 592,372 601,372 610,372 620,372 629,372 638,372 647,372 656,372 665,372 674,372 684,372 693,372 702,372 711,372 720,372 729,372 738,372 748,372 757,372 766,372 775,372 784,372 793,372 803,372 "/>
<polyline fill="none" opacity="1" stroke="#0000FF" stroke-width="3" points="71,372 80,369 89,365 98,358 107,350 116,340 125,329 135,317 144,303 153,288 162,273 171,258 180,245 189,234 199,227 208,222 217,220 226,219 235,219 244,219 254,220 263,221 272,222 281,222 290,222 299,222 308,223 318,223 327,223 336,223 345,223 354,223 363,222 372,222 382,222 391,222 400,222 409,222 418,222 427,222 437,222 446,222 455,222 464,222 473,222 482,222 491,222 501,222 510,222 519,222 528,222 537,222 546,222 555,222 565,222 574,222 583,222 592,222 601,222 610,222 620,222 629,222 638,222 647,222 656,222 665,222 674,222 684,222 693,222 702,222 711,222 720,222 729,222 738,222 748,222 757,222 766,222 775,222 784,222 793,222 803,222 "/>
<polyline fill="none" opacity="0.3" stroke="#FF0000" stroke-width="2" points="71,73 80,73 89,73 98,73 107,73 116,73 125,73 135,73 144,73 153,73 162,73 171,73 180,73 189,73 199,73 208,73 217,73 226,73 235,73 244,73 254,73 263,73 272,73 281,73 290,73 299,73 308,73 318,73 327,73 336,73 345,73 354,73 363,73 372,73 382,73 391,73 400,73 409,73 418,73 427,73 437,73 446,73 455,73 464,73 473,73 482,73 491,73 501,73 510,73 519,73 528,73 537,73 546,73 555,73 565,73 574,73 583,73 592,73 601,73 610,73 620,73 629,73 638,73 647,73 656,73 665,73 674,73 684,73 693,73 702,73 711,73 720,73 729,73 738,73 748,73 757,73 766,73 775,73 784,73 793,73 803,73 "/>
<polyline fill="none" opacity="1" stroke="#FF0000" stroke-width="3" points="71,73 80,76 89,80 98,87 107,95 116,105 125,116 135,128 144,142 153,157 162,172 171,187 180,200 189,211 199,218 208,223 217,225 226,226 235,226 244,226 254,225 263,224 272,223 281,223 290,223 299,222 308,222 318,222 327,222 336,222 345,222 354,222 363,222 372,222 382,222 391,222 400,222 409,222 418,222 427,222 437,222 446,222 455,222 464,222 473,222 482,222 491,222 501,222 510,222 519,222 528,222 537,222 546,222 555,222 565,222 574,222 583,222 592,222 601,222 610,222 620,222 629,222 638,222 647,222 656,222 665,222 674,222 684,222 693,222 702,222 711,222 720,222 729,222 738,222 748,222 757,222 766,222 775,222 784,222 793,222 803,222 "/>
 
 
 
No action, start -1.25
 
 
Evolved, start -1.25
 
 
No action, start +1.25
 
 
Evolved, start +1.25

## 5. What this run establishes

With the default settings, the search uses **3,200 evaluations × 4 episodes × 80 steps = 1,024,000 training simulation steps**. The 100 batches include initialization. The no-action baseline, held-out score, and four plotted rollouts are additional diagnostics outside the search budget.

The three genes are optimized between evaluations; they do not change within an episode. The experiment demonstrates fixed-topology neuroevolution, not lifetime learning, architecture search, or open-ended evolution. One successful seed is a demonstration, not a benchmark study.

### Experiments to try

- Change the root seed and compare runs at the same evaluation budget.
- Adjust the mutation scale. Does more exploration help early progress but limit fine adjustment?
- Change the number of offspring and adjust the batch count so total search evaluations stay fixed.
- Try asymmetric training starts, then freeze the controller before evaluating untouched starts.

Restart the kernel and run all cells for an independent repeat. Running the training cell alone adds another 100 batches to the existing search.

### Sources and implementation

- [Official book contents and exercise list](https://neuroevolutionbook.com/)
- [Public search implementation](../crates/nevo/src/algorithms/elitist.rs)
- [Target-reaching task](../crates/nevo/src/tasks/target_reach.rs)
- [Equivalent command-line example](../crates/nevo/examples/ch01_introduction.rs)

This notebook's prose, task, and code are original companion material. It imports nevo's public building blocks rather than duplicating them.

## 6. Apply the chapter’s population structure

The book’s §1.1 (pp. 3–4) keeps the best **20 of 100** candidates and creates **80 offspring from those 20**. Figure 1.2 (p. 5) connects this population loop to network evaluation in an environment.

The existing `Ga` can express that structure. `elite_count = 20` controls unchanged copies; `parent_pool_size = Some(20)` independently restricts who can mate. With `tournament_size = 1`, parents are sampled uniformly from the top 20, with replacement. Every generation reevaluates all 100 candidates, including the copied elites.

The following experiment reuses `TargetReach`. Uniform initialization in `[-1, 1)`, uniform crossover, Gaussian mutation with standard deviation 0.2, seed 42, and 32 batches are our explicit choices. The default per-gene mutation probability `1 / dimensions` mutates one gene **on average**, not exactly one gene per child as a literal reading of the chapter’s illustrative mutation description might suggest. Some children have no mutated genes and others have several.

This additional run uses **3,200 search evaluations**, including initialization, and scores held-out starts afterward. It demonstrates composition; one run does not establish which search method is better. It is also available as [ch01_population](../crates/nevo/examples/ch01_population.rs).


In [8]:
let mut population_task = TargetReach::default();
let mut population_config = nevo::algorithms::GaConfig::new(TargetReach::PARAMETERS, 42);
population_config.population = 100;
population_config.elite_count = 20;
population_config.parent_pool_size = Some(20);
population_config.tournament_size = 1;
population_config.crossover_probability = 1.0;
population_config.initial_bound = 1.0;
population_config.mutation_sigma = 0.2;
let mut population_search = nevo::algorithms::Ga::new(population_config)?;
for _ in 0..32 {
    step(&mut population_search, &mut population_task)?;
}
let population_best = population_search.best().cloned().ok_or("no selected controller")?;
let population_held_out = TargetReach::new(vec![-1.25, -0.75, 0.75, 1.25], population_task.steps())?;
println!("Population: {}; eligible parents: 20; copied elites: 20", population_search.parents().len());
println!("Batches: {}; search evaluations: {}", population_search.batches(), population_search.evaluations());
println!("Training loss: {:.9}", population_best.output);
println!("Held-out loss: {:.9}", population_held_out.value(&population_best.job.input)?);

Population: 100; eligible parents: 20; copied elites: 20


Batches: 32; search evaluations: 3200


Training loss: 0.036622373


Held-out loss: 0.071014781
